In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import datetime
import os

import mlflow
from mlflow.models.signature import infer_signature
from mlflow.tracking import MlflowClient

from data import split_train_test_panel
from models import evaluate_all
from preprocessor_model_pipeline import PpModelPl
from raw_data import load_raw_data, sample_tickers_dates
from utils import get_pipreqs_from_pyproject

In [ ]:
TARGET = "returns"
ENV = "test"
SAMPLE_TICKERS = ["AAPL", "AMZN"]
ROOTPATH = os.path.dirname(os.getcwd())
DATAPATH = os.path.join(ROOTPATH, "data")
MODELPATH = os.path.join(ROOTPATH, "models")
STATIC_PARS = {
    "lags": 3,  # number of lags to be included
    "steps": 5,  # steps ahead to be forecasted
}
DYN_PARS = {"CldrFeats": False, "ModReg": True}
EXP_PARS = STATIC_PARS | DYN_PARS
EXP_PARS

In [ ]:
# copied from main_training.py
def base_data_prep(
    datasource: str,
    localrun: str,
    env: str,
    use_sample_tickers_for_training: bool,
    sample_fdir: str | None = None,
) -> tuple:
    # load the raw data
    df_raw, access_date_str = load_raw_data(
        datasource=datasource,
        datapath=DATAPATH,
        localrun=localrun,
        env=env,
        user="nelgiriyewithana",
        datasetname="world-stock-prices-daily-updating",
    )
    # sample tickers and dates
    df, _ = sample_tickers_dates(
        df_raw,
        tickers=SAMPLE_TICKERS if use_sample_tickers_for_training else None,
        startdate=datetime.datetime.now() - datetime.timedelta(days=365 * 5 + 1),
        datasource=datasource,
        sample_fdir=sample_fdir,
        access_date_str=access_date_str,
    )
    # store_sampled_data_in_gcs(env, fpath)
    # Split train and test
    df_train, df_test = split_train_test_panel(df, train_ratio=0.8)
    return df, df_train, df_test

## Prepare raw data, instantiate the preprocessor-model pipeline and train

In [ ]:
# prepare the raw data
df, df_train, df_test = base_data_prep(
    datasource="yahoofinance",
    localrun=True,
    env=ENV,
    use_sample_tickers_for_training=True,
    sample_fdir=os.path.join(DATAPATH, f"rawdata_samples_{ENV}"),
)
# instantiate the pipeline
pp_model_pl = PpModelPl(
    target="returns",
    steps=EXP_PARS["steps"],
    date_col="Date",
    price_col="Close",
    ticker_col="Ticker",
    winsorize=True,
    q_low=0.01,
    q_high=0.99,
    lags=EXP_PARS["lags"],
    CldrFeats=EXP_PARS["CldrFeats"],
    ModReg=EXP_PARS["ModReg"],
)
# fit the pipeline
pp_model_pl.fit(df_train)

## Log model in in MLflow

In [ ]:
MODEL_ARTIFACT_FOLDER = "mlflow_models"
REGISTRY_NAME = "stocks_forecasting_tests"
CLIENT = MlflowClient()
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("test")

In [ ]:
with mlflow.start_run() as run:
    # pull matrices without changing your evaluator
    X_train, y_train = pp_model_pl.make_X_y(df_train)
    X_test, y_test = pp_model_pl.make_X_y(df_test)

    # evaluate on the inner estimator that consumes X/y
    scores = evaluate_all(
        pp_model_pl.estimator_, X_train, y_train, X_test, y_test, df, SAMPLE_TICKERS
    )

    mlflow.log_params(EXP_PARS)
    # log the metrics
    for cat, score_dict in scores.items():
        for metric, value in score_dict.items():
            mlflow.log_metrics({f"{cat}_{metric}": value})

    signature = infer_signature(df_train, pp_model_pl.predict(df_train))
    pip_reqs = get_pipreqs_from_pyproject(os.path.join(ROOTPATH, "pyproject.toml"))
    # Log the sklearn-flavor model (this serializes your custom estimator)
    info = mlflow.sklearn.log_model(
        sk_model=pp_model_pl,
        name=MODEL_ARTIFACT_FOLDER,
        pip_requirements=pip_reqs,
        input_example=df_train.head(80),
        signature=signature,
        # include the exact code your model needs at inference
        code_paths=[
            "../src/preprocessor_model_pipeline.py",  # file defining PpModelPl
            "../src/data.py",  # module providing build_features, create_X_y_multistep
            "../src/models.py",  # module providing create_xgbregressor_chain
        ],
    )
    print("Run ID:", run.info.run_id)
    print("Model URI:", info.model_uri)

## Export the mlflow model

In [ ]:
import os

from mlflow.artifacts import download_artifacts

RUN_NAME = "test"
localdir = os.path.join(MODELPATH, RUN_NAME)
os.makedirs(localdir, exist_ok=True)

download_artifacts(
    artifact_uri=f"runs:/{run.info.run_id}/{MODEL_ARTIFACT_FOLDER}", dst_path=localdir
)
print(f"model artifacts are stored in: {localdir}")